# Phi-4-mini - Structured Output Benchmark v2 - Phase 3 (Constrained Decoding)
- **Model**: `microsoft/Phi-4-mini-instruct`
- **Parameters**: 3.8B (dense)
- **Hardware**: RTX 4090
- **Phase 3**: 14 tasks x 3 decoding conditions (native, Outlines, XGrammar) at greedy decoding
- **Goal**: Does constrained decoding rescue small-model structural failures? Classify Phase-1/Phase-2 failures as structural (CD-fixable) vs semantic (CD-resistant). Phi-4-mini is a failing model of interest: at 3.8B it tied Qwen3-0.6B (93%) in Phase 1 rather than matching same-scale dense peers -- architecture > params. Compare the CD rescue pattern against Qwen3-0.6B to see whether the failure shape differs by architecture at matched schema-validity.
- **Conditions**:
  - `native` - unconstrained greedy (reproduces Phase 1 baseline)
  - `outlines` - Outlines JSON schema generation
  - `xgrammar` - XGrammar logits processor
- **Determinism**: All conditions use greedy decoding (temperature=0.0, do_sample=False) so results are directly comparable.
- **Special handling**: none -- standard chat template, AutoTokenizer, no special pad_token_id. NOTE: requires a recent transformers version (the model-loading cell upgrades it before loading).


In [ ]:
!pip install huggingface_hub

from huggingface_hub import login
login()

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import torch
import json
import time
import re
from datetime import datetime

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
!pip install --upgrade transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"Model: {MODEL_NAME}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
"""
Task definitions for structured output benchmark.

Each task has:
- name: task identifier
- prompt: the actual prompt to send to the model
- schema: the expected JSON schema for valid output
- evaluator: how to check correctness
"""

import json

# ============================================================
# TASK CATEGORY 1: Simple JSON Generation
# Generate a JSON object from a natural language description
# ============================================================

SIMPLE_JSON_TASKS = [
    {
        "id": "json_simple_person",
        "category": "json_generation",
        "difficulty": "easy",
        "prompt": "Generate a JSON object for a person with the following fields: name (string), age (number), email (string), and city (string). Use realistic values.",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "number"},
                "email": {"type": "string"},
                "city": {"type": "string"}
            },
            "required": ["name", "age", "email", "city"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_simple_product",
        "category": "json_generation",
        "difficulty": "easy",
        "prompt": "Create a JSON object for a product listing with: product_name (string), price (number), in_stock (boolean), and category (string).",
        "schema": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string"},
                "price": {"type": "number"},
                "in_stock": {"type": "boolean"},
                "category": {"type": "string"}
            },
            "required": ["product_name", "price", "in_stock", "category"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_nested_address",
        "category": "json_generation",
        "difficulty": "medium",
        "prompt": "Generate a JSON object for a user profile. It must have: name (string), age (number), and address (object with street, city, state, zip).",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "number"},
                "address": {
                    "type": "object",
                    "properties": {
                        "street": {"type": "string"},
                        "city": {"type": "string"},
                        "state": {"type": "string"},
                        "zip": {"type": "string"}
                    },
                    "required": ["street", "city", "state", "zip"],
                    "additionalProperties": False
                }
            },
            "required": ["name", "age", "address"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_array_orders",
        "category": "json_generation",
        "difficulty": "medium",
        "prompt": "Create a JSON array containing 3 order objects. Each order should have: order_id (string), items (array of strings), total (number), and status (string that must be one of: pending, shipped, delivered).",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "items": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "total": {"type": "number"},
                    "status": {"type": "string", "enum": ["pending", "shipped", "delivered"]}
                },
                "required": ["order_id", "items", "total", "status"],
                "additionalProperties": False
            },
            "minItems": 3,
            "maxItems": 3
        }
    },
    {
        "id": "json_complex_api",
        "category": "json_generation",
        "difficulty": "hard",
        "prompt": "Generate a JSON object representing an API response. It should have: status (number), message (string), data (object with users array, where each user has id, name, email, role where role is one of admin/user/moderator), and metadata (object with total_count, page, per_page).",
        "schema": {
            "type": "object",
            "properties": {
                "status": {"type": "number"},
                "message": {"type": "string"},
                "data": {
                    "type": "object",
                    "properties": {
                        "users": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "id": {"type": "number"},
                                    "name": {"type": "string"},
                                    "email": {"type": "string"},
                                    "role": {"type": "string", "enum": ["admin", "user", "moderator"]}
                                },
                                "required": ["id", "name", "email", "role"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["users"],
                    "additionalProperties": False
                },
                "metadata": {
                    "type": "object",
                    "properties": {
                        "total_count": {"type": "number"},
                        "page": {"type": "number"},
                        "per_page": {"type": "number"}
                    },
                    "required": ["total_count", "page", "per_page"],
                    "additionalProperties": False
                }
            },
            "required": ["status", "message", "data", "metadata"],
            "additionalProperties": False
        }
    }
]

# ============================================================
# TASK CATEGORY 2: Schema Adherence
# Given a schema, generate output that matches it
# ============================================================

SCHEMA_ADHERENCE_TASKS = [
    {
        "id": "schema_weather",
        "category": "schema_adherence",
        "difficulty": "easy",
        "prompt": "Output a valid JSON object matching this exact schema. Generate realistic weather data:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"location\":{\"type\":\"string\"},\"temperature\":{\"type\":\"number\"},\"unit\":{\"type\":\"string\",\"enum\":[\"celsius\",\"fahrenheit\"]},\"conditions\":{\"type\":\"string\"},\"humidity\":{\"type\":\"number\",\"minimum\":0,\"maximum\":100}},\"required\":[\"location\",\"temperature\",\"unit\",\"conditions\",\"humidity\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string"},
                "temperature": {"type": "number"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                "conditions": {"type": "string"},
                "humidity": {"type": "number", "minimum": 0, "maximum": 100}
            },
            "required": ["location", "temperature", "unit", "conditions", "humidity"],
            "additionalProperties": False
        }
    },
    {
        "id": "schema_database_record",
        "category": "schema_adherence",
        "difficulty": "medium",
        "prompt": "Generate a JSON object matching this schema representing a database record. Fill in realistic values:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"id\":{\"type\":\"string\",\"pattern\":\"^[a-f0-9]{8}$\"},\"created_at\":{\"type\":\"string\",\"format\":\"date-time\"},\"type\":{\"type\":\"string\",\"enum\":[\"customer\",\"vendor\",\"employee\"]},\"active\":{\"type\":\"boolean\"},\"tags\":{\"type\":\"array\",\"items\":{\"type\":\"string\"},\"maxItems\":5}},\"required\":[\"id\",\"created_at\",\"type\",\"active\",\"tags\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "id": {"type": "string", "pattern": "^[a-f0-9]{8}$"},
                "created_at": {"type": "string", "format": "date-time"},
                "type": {"type": "string", "enum": ["customer", "vendor", "employee"]},
                "active": {"type": "boolean"},
                "tags": {
                    "type": "array",
                    "items": {"type": "string"},
                    "maxItems": 5
                }
            },
            "required": ["id", "created_at", "type", "active", "tags"],
            "additionalProperties": False
        }
    },
    {
        "id": "schema_config_file",
        "category": "schema_adherence",
        "difficulty": "hard",
        "prompt": "Generate a valid JSON configuration object matching this schema for a web server config:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"server\":{\"type\":\"object\",\"properties\":{\"host\":{\"type\":\"string\"},\"port\":{\"type\":\"integer\",\"minimum\":1,\"maximum\":65535},\"ssl\":{\"type\":\"object\",\"properties\":{\"enabled\":{\"type\":\"boolean\"},\"cert_path\":{\"type\":\"string\"},\"key_path\":{\"type\":\"string\"}},\"required\":[\"enabled\"]}},\"required\":[\"host\",\"port\",\"ssl\"]},\"logging\":{\"type\":\"object\",\"properties\":{\"level\":{\"type\":\"string\",\"enum\":[\"debug\",\"info\",\"warn\",\"error\"]},\"file\":{\"type\":\"string\"},\"rotate\":{\"type\":\"boolean\"}},\"required\":[\"level\"]},\"cors\":{\"type\":\"object\",\"properties\":{\"enabled\":{\"type\":\"boolean\"},\"origins\":{\"type\":\"array\",\"items\":{\"type\":\"string\"}},\"methods\":{\"type\":\"array\",\"items\":{\"type\":\"string\",\"enum\":[\"GET\",\"POST\",\"PUT\",\"DELETE\",\"PATCH\"]}}},\"required\":[\"enabled\"]}},\"required\":[\"server\",\"logging\",\"cors\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "server": {
                    "type": "object",
                    "properties": {
                        "host": {"type": "string"},
                        "port": {"type": "integer"},
                        "ssl": {
                            "type": "object",
                            "properties": {
                                "enabled": {"type": "boolean"},
                                "cert_path": {"type": "string"},
                                "key_path": {"type": "string"}
                            },
                            "required": ["enabled"],
                            "additionalProperties": False
                        }
                    },
                    "required": ["host", "port", "ssl"],
                    "additionalProperties": False
                },
                "logging": {
                    "type": "object",
                    "properties": {
                        "level": {"type": "string", "enum": ["debug", "info", "warn", "error"]},
                        "file": {"type": "string"},
                        "rotate": {"type": "boolean"}
                    },
                    "required": ["level"],
                    "additionalProperties": False
                },
                "cors": {
                    "type": "object",
                    "properties": {
                        "enabled": {"type": "boolean"},
                        "origins": {"type": "array", "items": {"type": "string"}},
                        "methods": {
                            "type": "array",
                            "items": {"type": "string", "enum": ["GET", "POST", "PUT", "DELETE", "PATCH"]}
                        }
                    },
                    "required": ["enabled"],
                    "additionalProperties": False
                }
            },
            "required": ["server", "logging", "cors"],
            "additionalProperties": False
        }
    }
]

# ============================================================
# TASK CATEGORY 3: Function Calling Format
# Generate tool/function call format (OpenAI-style)
# ============================================================

FUNCTION_CALLING_TASKS = [
    {
        "id": "funcall_get_weather",
        "category": "function_calling",
        "difficulty": "easy",
        "prompt": "You are a helpful assistant with access to the following function:\n\n{\"name\": \"get_weather\", \"description\": \"Get current weather for a location\", \"parameters\": {\"type\": \"object\", \"properties\": {\"location\": {\"type\": \"string\", \"description\": \"City name\"}, \"unit\": {\"type\": \"string\", \"enum\": [\"celsius\", \"fahrenheit\"]}}, \"required\": [\"location\"]}}\n\nThe user asks: \"What's the weather like in San Francisco?\"\n\nRespond with a function call in JSON format using: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "arguments": {"type": "object", "additionalProperties": True}
            },
            "required": ["name", "arguments"],
            "additionalProperties": False
        },
        "expected_function": "get_weather",
        "expected_params_keys": ["location"]
    },
    {
        "id": "funcall_search_multi",
        "category": "function_calling",
        "difficulty": "medium",
        "prompt": "You are a helpful assistant with access to the following functions:\n\n1. {\"name\": \"search_web\", \"description\": \"Search the web for information\", \"parameters\": {\"type\": \"object\", \"properties\": {\"query\": {\"type\": \"string\"}, \"num_results\": {\"type\": \"integer\", \"default\": 10}}, \"required\": [\"query\"]}}\n\n2. {\"name\": \"send_email\", \"description\": \"Send an email\", \"parameters\": {\"type\": \"object\", \"properties\": {\"to\": {\"type\": \"string\"}, \"subject\": {\"type\": \"string\"}, \"body\": {\"type\": \"string\"}}, \"required\": [\"to\", \"subject\", \"body\"]}}\n\nThe user asks: \"Search for the best restaurants in NYC and email the results to john@example.com\"\n\nRespond with the appropriate function call(s) in JSON format. If multiple calls are needed, use an array. Use the format: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "arguments": {"type": "object", "additionalProperties": True}
                },
                "required": ["name", "arguments"],
                "additionalProperties": False
            },
            "minItems": 1,
            "maxItems": 2
        },
        "expected_functions": ["search_web", "send_email"],
        "expected_params_keys": [["query", "num_results"], ["to", "subject", "body"]]
    },
    {
        "id": "funcall_database_query",
        "category": "function_calling",
        "difficulty": "hard",
        "prompt": "You are a helpful assistant with access to the following function:\n\n{\"name\": \"query_database\", \"description\": \"Execute a SQL query on the database\", \"parameters\": {\"type\": \"object\", \"properties\": {\"query\": {\"type\": \"string\", \"description\": \"SQL query to execute\"}, \"database\": {\"type\": \"string\", \"enum\": [\"production\", \"staging\", \"analytics\"]}, \"limit\": {\"type\": \"integer\", \"default\": 100, \"maximum\": 1000}, \"format\": {\"type\": \"string\", \"enum\": [\"json\", \"csv\"], \"default\": \"json\"}}, \"required\": [\"query\", \"database\"]}}\n\nThe user asks: \"Get me the top 50 customers by revenue from the production database in CSV format\"\n\nRespond with a function call in JSON format using: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "arguments": {"type": "object", "additionalProperties": True}
            },
            "required": ["name", "arguments"],
            "additionalProperties": False
        },
        "expected_function": "query_database",
        "expected_params_keys": ["query", "database", "limit", "format"]
    }
]

# ============================================================
# TASK CATEGORY 4: Key-Value Extraction
# Extract structured info from unstructured text
# ============================================================

EXTRACTION_TASKS = [
    {
        "id": "extract_business_card",
        "category": "extraction",
        "difficulty": "easy",
        "prompt": "Extract the contact information from the following text into a JSON object with fields: name, phone, email, company, title.\n\nText: \"John Smith is a Senior Software Engineer at TechCorp Inc. You can reach him at john.smith@techcorp.com or call (555) 123-4567.\"",
        "expected_values": {
            "name": "John Smith",
            "phone": "(555) 123-4567",
            "email": "john.smith@techcorp.com",
            "company": "TechCorp Inc",
            "title": "Senior Software Engineer"
        },
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "phone": {"type": "string"},
                "email": {"type": "string"},
                "company": {"type": "string"},
                "title": {"type": "string"}
            },
            "required": ["name", "phone", "email", "company", "title"],
            "additionalProperties": False
        }
    },
    {
        "id": "extract_receipt",
        "category": "extraction",
        "difficulty": "medium",
        "prompt": "Extract receipt information from the following text into a JSON object with: store_name, date, items (array of objects with name and price), subtotal, tax, total.\n\nText: \"WALMART SUPERCENTER\nDate: 03/15/2026\nMilk 2% 1gal ........... $4.98\nSourdough Bread ........ $3.49\nOrganic Eggs 12ct ...... $5.99\nAvocados 3ct ........... $4.47\nSubtotal: $18.93\nTax (8.25%): $1.56\nTOTAL: $20.49\"",
        "expected_values": {
            "store_name": "WALMART SUPERCENTER",
            "date": "03/15/2026",
            "items": [
                {"name": "Milk 2% 1gal", "price": 4.98},
                {"name": "Sourdough Bread", "price": 3.49},
                {"name": "Organic Eggs 12ct", "price": 5.99},
                {"name": "Avocados 3ct", "price": 4.47}
            ],
            "subtotal": 18.93,
            "tax": 1.56,
            "total": 20.49
        },
        "schema": {
            "type": "object",
            "properties": {
                "store_name": {"type": "string"},
                "date": {"type": "string"},
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "price": {"type": "number"}
                        },
                        "required": ["name", "price"],
                        "additionalProperties": False
                    }
                },
                "subtotal": {"type": "number"},
                "tax": {"type": "number"},
                "total": {"type": "number"}
            },
            "required": ["store_name", "date", "items", "subtotal", "tax", "total"]
        }
    },
    {
        "id": "extract_api_log",
        "category": "extraction",
        "difficulty": "hard",
        "prompt": "Extract structured data from the following API log entry into JSON with: timestamp, method, path, status_code, response_time_ms, error (null if no error), and headers (object with content-type, x-request-id, user-agent).\n\nText: '[2026-05-05T10:23:45.678Z] POST /api/v2/users/authenticate -> 401 (145.3ms) | Headers: {\"content-type\": \"application/json\", \"x-request-id\": \"req-abc123def456\", \"user-agent\": \"MobileApp/3.2.1 (iOS 17.4)\"} | Error: Invalid credentials - email not verified'",
        "schema": {
            "type": "object",
            "properties": {
                "timestamp": {"type": "string"},
                "method": {"type": "string", "enum": ["GET", "POST", "PUT", "DELETE", "PATCH"]},
                "path": {"type": "string"},
                "status_code": {"type": "number"},
                "response_time_ms": {"type": "number"},
                "error": {"oneOf": [{"type": "string"}, {"type": "null"}]},
                "headers": {
                    "type": "object",
                    "properties": {
                        "content-type": {"type": "string"},
                        "x-request-id": {"type": "string"},
                        "user-agent": {"type": "string"}
                    },
                    "required": ["content-type", "x-request-id", "user-agent"],
                    "additionalProperties": False
                }
            },
            "required": ["timestamp", "method", "path", "status_code", "response_time_ms", "error", "headers"]
        }
    }
]

# ============================================================
# SYSTEM PROMPTS for different conditions
# ============================================================

SYSTEM_PROMPTS = {
    "basic": "You are a helpful assistant. Always respond with valid JSON. Do not include any text outside the JSON.",
    
    "structured": "You are a helpful assistant. When asked to generate structured output, you MUST respond with ONLY valid JSON. No markdown, no code blocks, no explanation - just the raw JSON. Ensure all required fields are present and types are correct.",
    
    "schema_given": "You are a helpful assistant. You will be given a JSON schema. Generate output that EXACTLY matches the schema. Respond with ONLY the JSON, nothing else. No markdown code fences. Validate your output mentally before responding."
}

# ============================================================
# Task lookup helpers
# ============================================================

def get_all_tasks():
    """Return all tasks combined."""
    return SIMPLE_JSON_TASKS + SCHEMA_ADHERENCE_TASKS + FUNCTION_CALLING_TASKS + EXTRACTION_TASKS


def get_tasks_by_category(category):
    """Return tasks filtered by category."""
    all_tasks = get_all_tasks()
    return [t for t in all_tasks if t["category"] == category]


def get_tasks_by_difficulty(difficulty):
    """Return tasks filtered by difficulty."""
    all_tasks = get_all_tasks()
    return [t for t in all_tasks if t.get("difficulty") == difficulty]


In [ ]:
# --- Chat template formatter ---
def format_messages(messages):
    """Format messages using the model's chat template.
    Override for model-specific behavior."""
    # Default (Phi-4-mini uses the standard chat template):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


In [ ]:
# --- JSON extractor ---
def extract_json(raw):
    """Extract JSON from model output, handling markdown fences and extra text."""
    text = raw.strip()
    
    # 1. Direct parse
    try:
        return json.loads(text)
    except:
        pass
    
    # 2. Strip markdown fences (FIXED regex)
    match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            pass
    
    # 3. Find JSON boundaries
    for start_char, end_char in [('{', '}'), ('[', ']')]:
        s = text.find(start_char)
        e = text.rfind(end_char)
        if s != -1 and e > s:
            try:
                return json.loads(text[s:e+1])
            except:
                pass
    return None

In [ ]:
# --- Model-specific generate kwargs ---
# do_sample / temperature are set per-call inside run_single_task_cd() (Phase 3
# always uses greedy, do_sample=False). Phi-4-mini needs no special pad_token_id.
BASE_GENERATE_KWARGS = {
    "max_new_tokens": 512,
}


In [ ]:
# --- Tokenizer/processor reference ---
# Set this to whichever you loaded
tok = tokenizer  # or tok = processor for Gemma

In [ ]:
# --- Install CD frameworks (run once) ---
%pip install outlines xgrammar

In [ ]:
# ============================================================
# PHASE 3: Constrained Decoding x Model Scale
#
# Goal: Does constrained decoding rescue small-model failures?
#       Which failures are structural (CD-fixable) vs semantic (CD-resistant)?
#
# Conditions:
#   native   - unconstrained greedy (reproduces Phase 1)
#   outlines - Outlines JSON schema generation
#   xgrammar - XGrammar logits processor
#
# All conditions use greedy decoding (temperature=0.0, do_sample=False)
# so results are deterministic and directly comparable.
# ============================================================

# --- Install CD frameworks (run once) ---
# IMPORTANT: use %pip (NOT !pip or a shell `pip install`). %pip always installs
# into the Python that THIS Jupyter kernel is running; !pip / SSH-shell pip often
# target a different interpreter, which is why "I installed it but it still says
# not installed" happens on vast.ai / Colab.
#
#   %pip install outlines xgrammar
#
# NOTE: this notebook targets Outlines v1+ API (Generator + from_transformers).
# If you still see a warning below after installing, the message prints the REAL
# exception so you can tell "wrong kernel" from a dependency/version conflict.

import sys as _sys

# --- CD framework imports ---
HAS_OUTLINES = False
HAS_XGRAMMAR = False

# Outlines v1 API:
#   import outlines                      -> outlines.from_transformers, outlines.Generator
#   from outlines.types import JsonSchema  -> schema output type for Generator
try:
    import outlines
    from outlines.types import JsonSchema
    HAS_OUTLINES = True
except Exception as _e:
    # Broad catch: outlines is often "installed" but fails to import because of a
    # transitive dependency (e.g. a transformers version mismatch). Surface it.
    print(f"Warning: could not import outlines ({type(_e).__name__}: {_e}).")
    print(f"  kernel python: {_sys.executable}")
    print('  Install into THIS kernel via a cell:  %pip install outlines xgrammar')

try:
    import xgrammar
    HAS_XGRAMMAR = True
except Exception as _e:
    print(f"Warning: could not import xgrammar ({type(_e).__name__}: {_e}).")
    print(f"  kernel python: {_sys.executable}")
    print('  Install into THIS kernel via a cell:  %pip install outlines xgrammar')


# --- CD-aware single task runner ---
def run_single_task_cd(task, run_num=1, temperature=0.0, decoder="native"):
    """
    Run a single benchmark task with optional constrained decoding.

    Args:
        task: task dict from task_definitions
        run_num: sample index
        temperature: 0.0 (all Phase 3 uses greedy for determinism)
        decoder: "native", "outlines", or "xgrammar"
    """

    # Build messages
    user_content = task["prompt"]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS["structured"]},
        {"role": "user", "content": user_content},
    ]
    text = format_messages(messages)

    # Configure generation kwargs
    generate_kwargs = dict(BASE_GENERATE_KWARGS)
    generate_kwargs["do_sample"] = False  # Phase 3 = greedy for comparability
    # Don't add `temperature` when do_sample=False: newer transformers flags it
    # as an invalid/ignored generation flag. Phase 3 is always greedy anyway.

    compile_ms = 0.0

    # ============== NATIVE (baseline) ==============
    if decoder == "native":
        inputs = tok(text=text, return_tensors="pt").to("cuda")
        input_len = inputs["input_ids"].shape[1]

        start = time.time()
        outputs = model.generate(**inputs, **generate_kwargs)
        elapsed = (time.time() - start) * 1000

        response = tok.decode(outputs[0][input_len:], skip_special_tokens=True)
        num_tokens = outputs.shape[1] - input_len

    # ============== OUTLINES (v1 API) ==============
    elif decoder == "outlines":
        if not HAS_OUTLINES:
            raise ImportError("outlines not installed")

        # outlines_model is wrapped once per notebook (see next cell)

        # Build a Generator for this schema (compiles the FSM here)
        compile_start = time.time()
        schema_json = json.dumps(task["schema"])
        generator = outlines.Generator(outlines_model, JsonSchema(schema_json))
        compile_ms = (time.time() - compile_start) * 1000

        # Generate (Outlines manages tokenization internally; kwargs pass through)
        gen_start = time.time()
        gen_kwargs = {k: v for k, v in generate_kwargs.items()
                      if k in ["max_new_tokens", "pad_token_id"]}
        response = generator(text, **gen_kwargs)
        elapsed = (time.time() - gen_start) * 1000

        # Count tokens (Outlines v1 returns a raw string)
        num_tokens = len(tok.encode(response))

    # ============== XGRAMMAR (0.2 API) ==============
    elif decoder == "xgrammar":
        if not HAS_XGRAMMAR:
            raise ImportError("xgrammar not installed")

        # xgrammar_compiler is built once per notebook (TokenizerInfo is
        # per-model: it reads the whole vocab). See the init cell.

        # Compile grammar from schema (per-task) + build the HF logits processor
        compile_start = time.time()
        schema_str = json.dumps(task["schema"])
        compiled_grammar = xgrammar_compiler.compile_json_schema(schema_str)
        xgrammar_processor = xgrammar.contrib.hf.LogitsProcessor(compiled_grammar)
        compile_ms = (time.time() - compile_start) * 1000

        # Generate with logits processor injected
        inputs = tok(text=text, return_tensors="pt").to("cuda")
        input_len = inputs["input_ids"].shape[1]

        start = time.time()
        outputs = model.generate(
            **inputs,
            logits_processor=[xgrammar_processor],
            **generate_kwargs
        )
        elapsed = (time.time() - start) * 1000

        response = tok.decode(outputs[0][input_len:], skip_special_tokens=True)
        num_tokens = outputs.shape[1] - input_len

    else:
        raise ValueError(f"Unknown decoder: {decoder}")

    # ============== VALIDATION (identical for all decoders) ==============
    json_valid = False
    schema_valid = False
    parsed = None
    error_msg = None

    try:
        parsed = extract_json(response)
        if parsed is not None:
            json_valid = True
        else:
            error_msg = "Could not extract valid JSON"
    except Exception as e:
        error_msg = f"JSON parse error: {e}"

    if json_valid:
        try:
            import jsonschema
            jsonschema.validate(instance=parsed, schema=task["schema"])
            schema_valid = True
        except Exception as e:
            schema_valid = False
            error_msg = f"Schema validation: {str(e)[:100]}"

    tokens_per_sec = round(num_tokens / (elapsed / 1000), 1) if elapsed > 0 else 0

    result = {
        "run": run_num,
        "temperature": temperature,
        "decoder": decoder,
        "task_id": task["id"],
        "category": task["category"],
        "difficulty": task["difficulty"],
        "json_valid": json_valid,
        "schema_valid": schema_valid,
        "latency_ms": round(elapsed, 1),
        "compile_ms": round(compile_ms, 1),
        "tokens_generated": num_tokens,
        "tokens_per_sec": tokens_per_sec,
        "response_raw": response,
        "response_parsed": parsed,
        "error": error_msg,
    }

    status = "\u2713" if schema_valid else ("~" if json_valid else "\u2717")
    compile_str = f" (compile: {compile_ms:.0f}ms)" if compile_ms > 0 else ""
    print(f"  {status} {task['id']:<25} [{decoder:<8}] Schema:{schema_valid}  {elapsed:.0f}ms{compile_str}  {tokens_per_sec}tok/s")
    if error_msg and not schema_valid:
        print(f"    Error: {error_msg[:80]}")

    return result


In [ ]:
# --- Initialize CD frameworks (run once per model) ---

if HAS_OUTLINES:
    print("Wrapping model for Outlines (v1 from_transformers)...")
    outlines_model = outlines.from_transformers(model, tok)
    print("\u2713 Outlines model ready")

if HAS_XGRAMMAR:
    print("Initializing XGrammar (TokenizerInfo + GrammarCompiler) ...")
    # TokenizerInfo is per-model (reads the full vocab); build it once.
    # Pass model.config.vocab_size so the token bitmask matches lm_head size
    # (some models pad vocab to a multiple of 32).
    tokenizer_info = xgrammar.TokenizerInfo.from_huggingface(
        tok, vocab_size=model.config.vocab_size
    )
    xgrammar_compiler = xgrammar.GrammarCompiler(tokenizer_info)
    print("\u2713 XGrammar compiler ready")


In [ ]:
# ============================================================
# PHASE 3 RUN: Constrained Decoding x 14 tasks
# ============================================================

PHASE3_DECODERS = ["native", "outlines", "xgrammar"]
phase3_results = []

all_tasks = get_all_tasks()

print(f"\nPHASE 3: {MODEL_NAME} - {len(all_tasks)} tasks x {len(PHASE3_DECODERS)} decoders")
print(f"Started: {datetime.now().isoformat()}")
print("=" * 70)

for decoder in PHASE3_DECODERS:
    print(f"\n--- Decoder: {decoder} ---")
    for task in all_tasks:
        result = run_single_task_cd(task, run_num=1, temperature=0.0, decoder=decoder)
        result["phase"] = 3
        result["model"] = MODEL_NAME
        phase3_results.append(result)

# Summary
print(f"\n{'=' * 70}")
print(f"PHASE 3 SUMMARY: {MODEL_NAME}")
print(f"{'Task ID':<25} {'native':>8} {'outlines':>8} {'xgrammar':>8}")
print("-" * 55)

from collections import defaultdict
p3_agg = defaultdict(dict)
for r in phase3_results:
    p3_agg[r["task_id"]][r["decoder"]] = r["schema_valid"]

for task_id in sorted(p3_agg.keys()):
    n = p3_agg[task_id].get("native", "")
    o = p3_agg[task_id].get("outlines", "")
    x = p3_agg[task_id].get("xgrammar", "")
    print(f"{task_id:<25} {'✓' if n else '✗':>8} {'✓' if o else '✗':>8} {'✓' if x else '✗':>8}")

# Overall rates per decoder
for dec in PHASE3_DECODERS:
    total = sum(1 for r in phase3_results if r["decoder"] == dec)
    passed = sum(1 for r in phase3_results if r["decoder"] == dec and r["schema_valid"])
    pct = 100 * passed / total if total else 0
    avg_ms = sum(r["latency_ms"] for r in phase3_results if r["decoder"] == dec) / total
    avg_compile = sum(r["compile_ms"] for r in phase3_results if r["decoder"] == dec) / total
    print(f"\n{dec:<10}: {passed}/{total} ({pct:.1f}%)  avg {avg_ms:.0f}ms/task  (compile {avg_compile:.0f}ms)")

print(f"\nFinished: {datetime.now().isoformat()}")


In [ ]:
# ============================================================
# EXPORT: Save Phase 3 results
#   - JSON: raw, preserves response_raw/response_parsed (content audit later)
#   - CSV : summary, one row per (task, decoder) - includes `decoder` column
#           so Phase 1/2/3 rows can be analyzed together
# ============================================================

import csv
from pathlib import Path

MODEL_SHORT = MODEL_NAME.split("/")[-1]
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

Path("results/v2/phase3").mkdir(parents=True, exist_ok=True)

# 1. Raw JSON
raw_path = f"results/v2/phase3/phase3_{MODEL_SHORT}_raw_{timestamp}.json"
export_results = []
for r in phase3_results:
    er = dict(r)
    if er.get("response_parsed") is not None:
        er["response_parsed"] = str(er["response_parsed"])
    export_results.append(er)
with open(raw_path, "w") as f:
    json.dump(export_results, f, indent=2, default=str)
print(f"Raw results saved: {raw_path}")

# 2. CSV summary
csv_path = f"results/v2/phase3/phase3_{MODEL_SHORT}.csv"
CSV_COLUMNS = [
    "model", "phase", "decoder", "temperature", "run", "task_id",
    "category", "difficulty", "json_valid", "schema_valid",
    "latency_ms", "compile_ms", "tokens_generated", "tokens_per_sec", "error",
]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(phase3_results)
print(f"CSV results saved: {csv_path}  ({len(phase3_results)} rows)")


In [ ]:
# --- Cleanup ---
del model, tokenizer
torch.cuda.empty_cache()
import gc; gc.collect()
